## 1. Configuração de Ambiente

Iniciei o processo definindo as variáveis de ambiente e os caminhos para o catálogo `medalhao_credit`. Optei por garantir a criação do schema `silver` logo no início com `CREATE SCHEMA IF NOT EXISTS` para garantir a idempotência do notebook (ou seja, ele pode ser rodado múltiplas vezes sem falhar).

In [0]:
%sql

CREATE CATALOG IF NOT EXISTS medalhao_credit;

In [0]:
%sql

USE CATALOG medalhao_credit;

CREATE SCHEMA IF NOT EXISTS silver_credit;

In [0]:
%sql
USE SCHEMA silver_credit;

### 1.1 Importação de Bibliotecas
Iniciei o desenvolvimento importando as funções essenciais do **PySpark**.
Selecionei especificamente as funções de transformação (`col`, `trim`, `regexp_replace`, etc.) e os tipos de dados (`StringType`, `LongType`, etc.) que seriam necessários para aplicar as regras de negócio, a limpeza de texto e a normalização do *schema* nas etapas seguintes.

In [0]:
from pyspark.sql.functions import col, trim, regexp_replace, initcap, when, lower, lit, udf
from pyspark.sql.types import StringType, IntegerType, LongType, TimestampType

## 2. Configuração e Leitura da Bronze
Defini as variáveis de ambiente `catalogo`, `bronze_db_name` e `silver_db_name` para organizar os caminhos do *Data Lake*.
Em seguida, realizei a leitura da tabela bruta (`df_bronze`). Como o arquivo original não possuía cabeçalho (gerando colunas genéricas como `_c0`), preparei o DataFrame para as transformações seguintes.

In [0]:
catalogo = "medalhao_credit"
bronze_db_name = "bronze_credit"
silver_db_name = "silver_credit"

volume_path = "/Volumes/workspace/default/data"

print(f"Ambiente configurado: {catalogo}.{silver_db_name}")

In [0]:
# Leitura da tabela Bronze de Chamados
df_bronze = spark.table(f"{catalogo}.{bronze_db_name}.chamados")

display(df_bronze.limit(5))

In [0]:
# Célula de Diagnóstico
print("Lista exata de colunas:")
print(df_bronze.columns)

%md
## 3. Tratamento de Identificadores (IDs)
Nesta etapa, foquei na chave primária da tabela. Renomeei a coluna genérica `_c0` para `id_chamado`, seguindo as boas práticas de *snake_case*.
Para garantir a integridade dos dados, converti o campo para o tipo Inteiro (`int`) e apliquei a remoção de duplicatas (`dropDuplicates`), assegurando que cada chamado seja único na camada Silver.

In [0]:
df_ordenado = (
    df_bronze
    .withColumnRenamed("_c0", "id_chamado") #Renomeia a coluna
    .withColumn("id_chamado", col("id_chamado").cast("int")) #Transforma tudo em int
    .filter(col("id_chamado").isNotNull())  #Remove linhas se o ID estiver vazio (lixo)
    .dropDuplicates(["id_chamado"])         #Se tiver dois IDs iguais, mantém apenas um
    .orderBy("id_chamado")
)

display(df_ordenado)

## 4. Normalização e Tipagem do ID Cliente
No dataframe `df_cliente_tratado`, renomeei a coluna para `id_cliente` (padrão *snake_case*).
Optei pela tipagem `long` para preservar a integridade de números grandes (como CPFs) e tratei os valores nulos preenchendo com `-1` (Cliente Desconhecido), garantindo que nenhum chamado fosse descartado por falta de identificação do cliente.

In [0]:
df_cliente_tratado = (
    df_ordenado # Continuando do passo anterior
    
    # 1. Renomear
    .withColumnRenamed("_c1", "id_cliente")
    
    # 2. Tipagem SEGURA (Long em vez de Int para não quebrar CPFs)
    .withColumn("id_cliente", col("id_cliente").cast("long"))
    
    # 3. Tratamento de Nulos (Regra de Ouro)
    # Não apaga a linha (o chamado existiu), mas marca o cliente como -1 (Desconhecido)
    .fillna(-1, subset=["id_cliente"])
)

display(df_cliente_tratado)

## 5. Reconstrução da Coluna Motivo
No dataframe `df_motivo_tratado`, corrigi os erros de *encoding* (ex: "Contrata..o") utilizando Expressões Regulares (*Regex*).
Substituí os padrões corrompidos pelas palavras corretas e refinei a regra da palavra "Não" (usando `\b` para limites de palavra), evitando alterações indevidas em palavras como "Pontos". Finalizei padronizando o texto com a primeira letra maiúscula (*Initcap*).

In [0]:
df_motivo_tratado = (
    df_cliente_tratado # Continua do passo de ID_Cliente
    
    # 1. Renomear
    .withColumnRenamed("_c2", "motivo")
    
    # 2. Tipagem e Trim (Limpeza básica)
    .withColumn("motivo", trim(col("motivo").cast("string")))
    
    # 3. CIRURGIA DE RECONSTRUÇÃO (Regex)
    # O ponto (.) substitui o caractere estragado. 
    
    .withColumn("motivo", regexp_replace(col("motivo"), "Contrata..o", "Contratacao"))
    .withColumn("motivo", regexp_replace(col("motivo"), "Contesta..o", "Contestacao"))
    .withColumn("motivo", regexp_replace(col("motivo"), "Altera..o", "Alteracao"))
    .withColumn("motivo", regexp_replace(col("motivo"), "cart.o", "cartao"))
    .withColumn("motivo", regexp_replace(col("motivo"), "D.vidas", "Duvidas"))
    .withColumn("motivo", regexp_replace(col("motivo"), "Informa..es", "Informacoes"))
    .withColumn("motivo", regexp_replace(col("motivo"), "Solicita..o", "Solicitacao"))

    # \\b significa "borda da palavra". Só pega se começar e terminar ali.
    .withColumn("motivo", regexp_replace(col("motivo"), "(?i)\\bn.o\\b", "Nao"))
    
    # 4. Padronização Visual (Capitalize)
    # Deixa "duvidas gerais" -> "Duvidas Gerais"
    .withColumn("motivo", initcap(col("motivo")))
    
    # 5. Tratamento de Nulos
    # Regra: Motivo vazio vira "Motivo Nao Informado"
    .fillna("Motivo Nao Informado", subset=["motivo"])
    .withColumn("motivo", 
                when((col("motivo") == "") | (col("motivo").isNull()), "Motivo Nao Informado")
                .otherwise(col("motivo")))
)

display(df_motivo_tratado)

## 6. Padronização de Canais
No dataframe `df_canal_tratado`, normalizei a escrita dos canais de atendimento (unificando "U.r.a" e "URA").
Implementei uma lógica hierárquica de regras (`when/otherwise`), priorizando a identificação de "Atendimento Especializado" antes de "Inicial" para evitar erros de classificação por *substrings*.

In [0]:
df_canal_tratado = (
    df_motivo_tratado # Continua do passo anterior
    
    # 1. Renomear
    .withColumnRenamed("_c3", "canal")
    
    # 2. Tipagem e Trim
    .withColumn("canal", trim(col("canal").cast("string")))
    
    # 3. NORMALIZAÇÃO E PADRONIZAÇÃO
    .withColumn("canal", 
                
                # Regra 1: Chatbot
                when(lower(col("canal")).like("%chat%"), "Chatbot")
                
                # Regra 2: URA (Pega URA ou U.r.a)
                .when((lower(col("canal")).like("%ura%")) | (lower(col("canal")).like("%u.r.a%")), "URA")
                
                # Regra 3: Web e Email (Adicionei conforme sua lista)
                .when(lower(col("canal")).like("%web%"), "Web")
                .when(lower(col("canal")).like("%mail%"), "Email")
                
                # Regra 4: ATENDIMENTO ESPECIALIZADO (Checa ANTES do Inicial)
                # Se tiver a palavra "especializado" em qualquer lugar, classifica aqui
                .when(lower(col("canal")).like("%especializ%"), "Atendimento Especializado")
                
                # Regra 5: ATENDIMENTO INICIAL
                # Pega "Inicial", "Atend. Inicial", ou qualquer "Atend" genérico que sobrou
                .when((lower(col("canal")).like("%inici%")) | (lower(col("canal")).like("%atend%")), "Atendimento Inicial")
                
                .otherwise(initcap(col("canal")))
               )

    # 4. Tratamento de Nulos
    .fillna("Canal Nao Identificado", subset=["canal"])
    .withColumn("canal", 
                when((col("canal") == "") | (col("canal").isNull()), "Canal Nao Identificado")
                .otherwise(col("canal")))
)

# Validação Final
print("Validação: Verifique se Especializado e Inicial estão separados:")
df_canal_tratado.groupBy("canal").count().show(truncate=False)

display(df_canal_tratado)

## 7. Flag Binária de Resolução
No dataframe `df_resolvido_tratado`, transformei a coluna de status em uma flag binária limpa.
Em vez de tratar acentos individualmente, usei uma lógica robusta que verifica a presença das letras "s" ou "n" (`like %s%`), blindando o código contra variações de escrita como "Sim", "SIM" ou erros de caracteres no "Não".

In [0]:
df_resolvido_tratado = (
    df_canal_tratado # Continua do passo anterior (Canal)
    
    # 1. Renomear (Snake Case)
    .withColumnRenamed("_c4", "resolvido")
    
    # 2. Tipagem e Trim
    .withColumn("resolvido", trim(col("resolvido").cast("string")))
    
    # 3. NORMALIZAÇÃO BINÁRIA (melhor prática)
    # Estratégia: Em vez de brigar com o acento, usamos a lógica do "Contém S"
    .withColumn("resolvido", 
                
                # Regra 1: Se tiver "s" ou "S" (Sim, S, yes), vira "Sim"
                when(lower(col("resolvido")).like("%s%"), "Sim")
                
                # Regra 2: Se tiver "n" ou "N" (Nao, No, No), vira "Nao"
                .when(lower(col("resolvido")).like("%n%"), "Nao")
                
                # Caso contrário (Vazio ou Lixo), vira "Nao Informado"
                .otherwise("Nao Informado")
               )

    # 4. Tratamento de Nulos (Garantia Extra)
    # Se sobrar algum null real, vira "Nao Informado"
    .fillna("Nao Informado", subset=["resolvido"])
)

# Validação: Deve aparecer APENAS "Sim", "Nao" e talvez "Nao Informado"
print("Distribuição da coluna Resolvido:")
df_resolvido_tratado.groupBy("resolvido").count().show()

display(df_resolvido_tratado)

## 8. Estruturação da Hora de Abertura
Nesta etapa, tratei exclusivamente a coluna `hora_abertura_chamado` (antiga `_c5`).
Embora os dados atuais estivessem vazios ou nulos, decidi forçar a tipagem imediata para `Timestamp` (*Schema Enforcement*). Essa decisão garante que a tabela Silver nasça com a estrutura correta de "Data e Hora" para receber dados futuros, evitando que a coluna permaneça como um texto genérico indefinido.

In [0]:
df_hora_abertura_tratado = (
    df_resolvido_tratado # Continua do passo anterior
    
    # 1. Renomear (Snake Case e Descritivo)
    .withColumnRenamed("_c5", "hora_abertura_chamado")
    
    # 2. Tipagem Forte (Schema Enforcement)
    # Mesmo que esteja tudo Null ou vazio, o tipo é Timestamp.
    .withColumn("hora_abertura_chamado", col("hora_abertura_chamado").cast("timestamp"))
)

# Validação (quero ver o schema como "timestamp" e os dados como "null")
print("Schema da coluna:")
df_hora_abertura_tratado.select("hora_abertura_chamado").printSchema()

print("\nVisualização dos dados (Devem estar null):")
df_hora_abertura_tratado.select("hora_abertura_chamado").show(5)

display(df_hora_abertura_tratado)

## 9. Estruturação Temporal e Regra de Cópia
No dataframe `df_inicio_tratado`, forcei a tipagem das colunas de horário para `Timestamp`.
Implementei a regra de negócio para a **Hora de Início**: quando o registro indicava "igual a hora de abertura", o código copiou dinamicamente o valor da coluna anterior.

In [0]:
df_inicio_tratado = (
    df_hora_abertura_tratado # Continua do passo anterior
    
    # 1. Renomear
    .withColumnRenamed("_c6", "hora_inicio_atendimento")
    
    # 2. Trim (Limpeza de espaços)
    .withColumn("hora_inicio_atendimento", trim(col("hora_inicio_atendimento")))
    
    # 3. Lógica de Negócio (A cópia condicional)
    # Se o texto disser "igual...", ele busca o valor da coluna hora_abertura_chamado.
    .withColumn("hora_inicio_atendimento", 
                when(lower(col("hora_inicio_atendimento")).like("%igual%"), col("hora_abertura_chamado"))
                .otherwise(col("hora_inicio_atendimento")))
    
    # 4. Tipagem Final (Schema Enforcement)
    # Tudo que não for data válida vira Null automaticamente aqui
    .withColumn("hora_inicio_atendimento", col("hora_inicio_atendimento").cast("timestamp"))
)

# Validação:
print("Schema atualizado:")
df_inicio_tratado.select("hora_inicio_atendimento").printSchema()

display(df_inicio_tratado)

## 10. Estruturação da Hora de Finalização
Nesta etapa, tratei a coluna `hora_finalizacao_atendimento` (antiga `_c7`).
Realizei a limpeza de espaços em branco (*trim*) e forcei a conversão direta para o tipo `Timestamp`. Diferente da hora de início, não houve necessidade de regras condicionais complexas, mas a definição estrita do tipo garante que a tabela esteja tecnicamente preparada para receber os registros de tempo assim que estiverem disponíveis na origem.

In [0]:
df_fim_tratado = (
    df_inicio_tratado # Continua do passo anterior
    
    # 1. Renomear
    .withColumnRenamed("_c7", "hora_finalizacao_atendimento")
    
    # 2. Trim (Limpeza básica de espaços invisíveis)
    .withColumn("hora_finalizacao_atendimento", trim(col("hora_finalizacao_atendimento")))
    
    # 3. Tipagem (Schema Enforcement)
    # Transforma texto/vazio em Data Real.
    .withColumn("hora_finalizacao_atendimento", col("hora_finalizacao_atendimento").cast("timestamp"))
)

# Validação do Schema
print("Schema final das colunas de tempo:")
df_fim_tratado.select("hora_abertura_chamado", 
                      "hora_inicio_atendimento", 
                      "hora_finalizacao_atendimento").printSchema()

display(df_fim_tratado)

## 11. Sanitização do Tempo de Espera
No dataframe `df_espera_tratado`, tratei a métrica `tempo_espera_segundos`.
Identifiquei que a origem enviava a string "NULL", então apliquei uma sanitização para converter esse texto em nulo real antes da tipagem para `int`. Assumi valores nulos como `0` e corrigi eventuais números negativos para garantir a consistência dos cálculos de média futuros.

In [0]:
df_espera_tratado = (
    df_fim_tratado # Continua do passo anterior
    
    # 1. Renomear (coloquei _segundos pra especificar o tipo de tempo)
    .withColumnRenamed("_c8", "tempo_espera_segundos")
    
    # 2. Trim (Limpeza de espaços)
    .withColumn("tempo_espera_segundos", trim(col("tempo_espera_segundos")))
    
    # 3. SANITIZAÇÃO 
    # Antes de converter para número, removi a palavra "NULL" e vazios
    .withColumn("tempo_espera_segundos", 
                when((col("tempo_espera_segundos") == "NULL") | (col("tempo_espera_segundos") == ""), None)
                .otherwise(col("tempo_espera_segundos")))
    
    # 4. Tipagem
    .withColumn("tempo_espera_segundos", col("tempo_espera_segundos").cast("int"))
    
    # 5. Negativos viram 0
    .withColumn("tempo_espera_segundos", 
                when(col("tempo_espera_segundos") < 0, 0)
                .otherwise(col("tempo_espera_segundos")))
    
    # Nulos viram 0 para cálculo de média
    .fillna(0, subset=["tempo_espera_segundos"])
)

# Validação
print("Estatísticas do Tempo de Espera (Segundos):")
df_espera_tratado.select("tempo_espera_segundos").describe().show()

display(df_espera_tratado)

## 12. Sanitização do Tempo de Conversa
No dataframe `df_conversa_tratado`, apliquei a mesma lógica de limpeza na coluna `tempo_conversa_segundos`.
Mantive os registros com duração "1" (mesmo sem *timestamps* válidos), preservando a informação de chamadas rápidas para análises de "Chamadas Fantasmas" na camada Gold.

In [0]:
df_conversa_tratado = (
    df_espera_tratado # Continua do passo anterior (Espera)
    
    # 1. Renomear
    .withColumnRenamed("_c9", "tempo_atendimento_segundos")
    
    # 2. Trim
    .withColumn("tempo_atendimento_segundos", trim(col("tempo_atendimento_segundos")))
    
    # 3. SANITIZAÇÃO (O Fix do "NULL" string)
    # Removemos a palavra escrita "NULL" antes de converter
    .withColumn("tempo_atendimento_segundos", 
                when((col("tempo_atendimento_segundos") == "NULL") | (col("tempo_atendimento_segundos") == ""), None)
                .otherwise(col("tempo_atendimento_segundos")))
    
    # 4. Tipagem (Integer)
    .withColumn("tempo_atendimento_segundos", col("tempo_atendimento_segundos").cast("int"))
    
    # 5. Regras de Sanidade
    # Negativos viram 0
    .withColumn("tempo_atendimento_segundos", 
                when(col("tempo_atendimento_segundos") < 0, 0)
                .otherwise(col("tempo_atendimento_segundos")))
    
    # Nulos viram 0 (Assumo zero conversa se estiver vazio)
    .fillna(0, subset=["tempo_atendimento_segundos"])
)

# Validação
print("Estatísticas do Tempo de Conversa (Segundos):")
df_conversa_tratado.select("tempo_atendimento_segundos").describe().show()

display(df_conversa_tratado)

## 13. ID Atendente e Carga Final
No dataframe `df_final`, tratei a coluna `id_atendente`. Preenchi os valores nulos com `-1`, criando a categoria "Atendimento Automático" para evitar perdas em cruzamentos com a tabela de funcionários.
Por fim, gravei o resultado na tabela `chamados` do banco `silver_db_name`, utilizando o formato Delta com sobrescrita (`mode("overwrite")`) para atualizar a camada Silver.

In [0]:
df_final = (
    df_conversa_tratado # Continua do passo anterior (Tempo Conversa)
    
    # 1. Renomear (Snake Case)
    .withColumnRenamed("_c10", "id_atendente")
    
    # 2. Trim
    .withColumn("id_atendente", trim(col("id_atendente")))
    
    # 3. SANITIZAÇÃO (Limpa a string "NULL" e vazios)
    .withColumn("id_atendente", 
                when((col("id_atendente") == "NULL") | (col("id_atendente") == ""), None)
                .otherwise(col("id_atendente")))
    
    # 4. Tipagem (Integer)
    .withColumn("id_atendente", col("id_atendente").cast("int"))
    
    # 5. Tratamento de Nulos
    # Se estiver vazio, coloquei -1 (Indica URA/Bot ou erro de sistema)
    .fillna(-1, subset=["id_atendente"])
)

# Validação
print("Amostra dos IDs de Atendente:")
df_final.select("id_atendente").distinct().show(10)

# --- GRAVAÇÃO FINAL DA TABELA SILVER ---
nome_tabela_silver = f"{catalogo}.{silver_db_name}.chamados"

(
    df_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(nome_tabela_silver)
)

print(f"Tabela salva em: {nome_tabela_silver}")
display(df_final)